# 02. Salmon quantification and gene count matrix

このノートブックは、FASTQ readを発現量テーブルへ変換する段階である。

**この段階が答える問い**

- 各サンプルで、どのtranscriptがどれくらい読まれたか。
- transcript単位の結果をgene単位にまとめると、どんなcount matrixになるか。

**入力**

- `metadata/samples.tsv`
- FASTQファイル
- `reference/gencode_grch38/salmon_index/`
- `reference/gencode_grch38/tx2gene.tsv`

**出力**

- `results/quant/salmon/<sample_id>/quant.sf`
- `results/counts/gene_counts.tsv`

**次に進む条件**

- 9サンプルすべてに `quant.sf` がある。
- gene count matrix の列名が `samples.tsv` の `sample_id` と一致している。


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

import pandas as pd

PROJECT_DIR = Path("/Users/yusuke_tateishi/Documents/RNA_seq").resolve()
CONFIG = json.loads((PROJECT_DIR / "config" / "analysis_config.json").read_text(encoding="utf-8"))
SAMPLES = pd.read_csv(PROJECT_DIR / CONFIG["samples_path"], sep="\t")

QUANT_DIR = PROJECT_DIR / "results" / "quant" / "salmon"
COUNTS_DIR = PROJECT_DIR / "results" / "counts"
QUANT_DIR.mkdir(parents=True, exist_ok=True)
COUNTS_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES[["sample_id", "condition", "fastq_1", "fastq_2"]]


## 参照ファイルが準備済みか確認する

ここが `False` なら、先に `00b_reference_setup_gencode_grch38.ipynb` を実行する。


In [ ]:
def resolve_project_path(path_text: str) -> Path:
    path = Path(path_text)
    return path if path.is_absolute() else PROJECT_DIR / path

reference = CONFIG["reference"]
reference_paths = {
    "salmon_index": resolve_project_path(reference["salmon_index"]),
    "tx2gene_path": resolve_project_path(reference["tx2gene_path"]),
    "transcript_fasta": resolve_project_path(reference.get("transcript_fasta", "")) if reference.get("transcript_fasta") else None,
    "gtf_path": resolve_project_path(reference["gtf_path"]) if reference.get("gtf_path") else None,
}

readiness = {
    name: (path.exists() if path is not None else False)
    for name, path in reference_paths.items()
}
readiness


## Salmonコマンドを作る

`--libType A` は、Salmonにライブラリの向きを自動推定させる設定である。初心者の最初の解析ではこの設定が扱いやすい。


In [ ]:
THREADS = int(CONFIG["quantification"].get("threads", 8))
LIBRARY_TYPE = CONFIG["quantification"].get("salmon", {}).get("library_type", "A")

salmon_commands = []
for _, sample in SAMPLES.iterrows():
    output_dir = QUANT_DIR / sample["sample_id"]
    command = [
        "salmon",
        "quant",
        "-i",
        str(reference_paths["salmon_index"]),
        "-l",
        LIBRARY_TYPE,
        "-1",
        str(PROJECT_DIR / sample["fastq_1"]),
        "-2",
        str(PROJECT_DIR / sample["fastq_2"]),
        "-p",
        str(THREADS),
        "--validateMappings",
        "-o",
        str(output_dir),
    ]
    salmon_commands.append(command)

command_path = QUANT_DIR / "salmon_commands.txt"
command_path.write_text("\n\n".join(" ".join(command) for command in salmon_commands) + "\n", encoding="utf-8")
print("Commands written to:", command_path.relative_to(PROJECT_DIR))
print("First command:")
print(" ".join(salmon_commands[0]))


## Salmonを実行する

初回だけ `RUN_SALMON = True` にする。処理時間はデータ量とCPU数に依存する。


In [ ]:
RUN_SALMON = False

if RUN_SALMON:
    salmon = shutil.which("salmon")
    if salmon is None:
        raise RuntimeError("salmon was not found. Activate the rna-seq environment first.")
    if not reference_paths["salmon_index"].exists():
        raise FileNotFoundError(reference_paths["salmon_index"])
    for command in salmon_commands:
        command[0] = salmon
        print("Running:", " ".join(command))
        subprocess.run(command, check=True)
else:
    print("Set RUN_SALMON = True after the Salmon index is ready.")


## Salmon結果がそろっているか確認する

各サンプルの `quant.sf` が存在すれば、次にgene単位へ集計できる。


In [ ]:
quant_status = []
for sample_id in SAMPLES["sample_id"]:
    quant_path = QUANT_DIR / sample_id / "quant.sf"
    quant_status.append({"sample_id": sample_id, "quant_sf_exists": quant_path.exists(), "path": str(quant_path.relative_to(PROJECT_DIR))})

quant_status = pd.DataFrame(quant_status)
quant_status


## gene count matrixを作る

Salmonの `quant.sf` にはtranscript単位の `NumReads` が入っている。ここでは `tx2gene.tsv` を使ってgene単位に合計する。

これはDESeq2へ渡すための表である。行がgene、列がsampleになる。


In [ ]:
BUILD_GENE_COUNTS_FROM_SALMON = False

if BUILD_GENE_COUNTS_FROM_SALMON:
    tx2gene_path = reference_paths["tx2gene_path"]
    if not tx2gene_path.exists():
        raise FileNotFoundError(tx2gene_path)

    tx2gene = pd.read_csv(tx2gene_path, sep="\t")
    required_columns = {"Name", "gene_id"}
    if not required_columns.issubset(tx2gene.columns):
        raise ValueError(f"tx2gene.tsv must contain columns: {required_columns}")
    tx2gene = tx2gene[["Name", "gene_id"]].drop_duplicates()

    per_sample_counts = []
    for sample_id in SAMPLES["sample_id"]:
        quant_path = QUANT_DIR / sample_id / "quant.sf"
        if not quant_path.exists():
            raise FileNotFoundError(quant_path)
        quant = pd.read_csv(quant_path, sep="\t", usecols=["Name", "NumReads"])
        merged = quant.merge(tx2gene, on="Name", how="inner")
        gene_counts = merged.groupby("gene_id", as_index=True)["NumReads"].sum()
        gene_counts.name = sample_id
        per_sample_counts.append(gene_counts)

    count_matrix = pd.concat(per_sample_counts, axis=1).fillna(0).round().astype(int)
    count_matrix.insert(0, "gene_id", count_matrix.index)
    out_path = PROJECT_DIR / CONFIG["differential_expression"]["count_matrix_path"]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    count_matrix.to_csv(out_path, sep="\t", index=False)
    print("Wrote:", out_path.relative_to(PROJECT_DIR))
    display(count_matrix.head())
else:
    print("Set BUILD_GENE_COUNTS_FROM_SALMON = True after all quant.sf files exist.")


## count matrixの形を確認する

count matrixが既にあれば、行数・列数・列名を確認する。


In [ ]:
count_path = PROJECT_DIR / CONFIG["differential_expression"]["count_matrix_path"]
if count_path.exists():
    counts = pd.read_csv(count_path, sep="\t")
    print("shape:", counts.shape)
    print("columns:", list(counts.columns[:10]))
    display(counts.head())
    missing_samples = set(SAMPLES["sample_id"]) - set(counts.columns)
    extra_samples = set(counts.columns) - {"gene_id"} - set(SAMPLES["sample_id"])
    print("missing_samples:", sorted(missing_samples))
    print("extra_samples:", sorted(extra_samples))
else:
    print("No gene count matrix yet:", count_path.relative_to(PROJECT_DIR))


**よくある間違い**

- `00b` を飛ばして `salmon_index` がない。
- `quant.sf` が一部のサンプルだけ存在する。
- `tx2gene.tsv` のtranscript IDと `quant.sf` の `Name` が合わず、集計後のgene数が少なすぎる。

**小さい練習**

`quant_status` で `False` のサンプルがあれば、そのサンプルのSalmonログを確認しよう。
